# 07 · Kernels regulares y desigualdad de accesibilidad — Bogotá 2023

**Objetivo.** Estimar diferencias por sexo y estrato socioeconómico en la accesibilidad estocástica hacia trabajo, educación y salud, evitando que la escasez de transiciones de un subgrupo produzca cadenas desconectadas o resultados infinitos.

Para cada grupo (g) y origen (i), el kernel empírico se contrae hacia el kernel poblacional:

\[
\widetilde P^{(g)}_{i\cdot}
=\omega_{ig}P^{(g)}_{i\cdot}+(1-\omega_{ig})P^{(0)}_{i\cdot},
\qquad
\omega_{ig}=\frac{n^{\mathrm{eff}}_{ig}}{n^{\mathrm{eff}}_{ig}+\tau}.
\]

La especificación principal usa (\\tau=20), con sensibilidad a 10 y 50. Los intervalos se obtienen mediante bootstrap Poisson por hogar. El estimando principal emplea la misma distribución poblacional de orígenes para todos los grupos: compara kernels de movilidad sin mezclar todavía composición residencial.


In [ ]:
%pip -q install pandas pyarrow numpy scipy matplotlib seaborn


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import os
import re
import shutil
import unicodedata
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

ROOT = Path("/content/colombia-stochastic-access")
RAW = ROOT / "data/raw/bogota_em2023"
INTERIM = ROOT / "data/interim/bogota_em2023_gate8"
OUT = ROOT / "outputs/bogota_em2023/gate8"
DRIVE_BASE = Path(
    "/content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023"
)
for directory in (RAW, INTERIM, OUT):
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_URL = (
    "https://observatorio.movilidadbogota.gov.co/sites/default/files/"
    "2025-09/05_Base%20datos%20procesada%20EODH.zip"
)
PURPOSES = ("employment", "education", "health")
TARGET_DEFINITIONS = ("core_50", "extended_80")
PRIMARY_TAU = 20.0
TAU_GRID = (10.0, 20.0, 50.0)
BOOTSTRAP_REPLICATES = 50
RANDOM_SEED = 202309

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"Entorno fuera de Colab o Drive no montado: {exc}")

gate4_dir = Path(os.environ.get("GATE4_DIR_OVERRIDE", DRIVE_BASE / "results/gate4"))
gate6_dir = Path(os.environ.get("GATE6_DIR_OVERRIDE", DRIVE_BASE / "results/gate6"))
gate7_dir = Path(os.environ.get("GATE7_DIR_OVERRIDE", DRIVE_BASE / "results/gate7"))
EDGE_PATH = gate4_dir / "transition_edges_utam.parquet"
TARGET_PATH = gate6_dir / "purpose_target_sets.csv"
GATE7_PATH = gate7_dir / "gate_7.json"

missing = [str(path) for path in (EDGE_PATH, TARGET_PATH, GATE7_PATH) if not path.exists()]
if missing:
    raise FileNotFoundError("Faltan insumos validados: " + ", ".join(missing))

gate7 = json.loads(GATE7_PATH.read_text(encoding="utf-8"))
gate7_status = gate7.get("gate", gate7)
if not bool(gate7_status.get("gate_7_passed", False)):
    raise RuntimeError("La Compuerta 7 no fue aprobada.")

print("Kernel poblacional:", EDGE_PATH)
print("Destinos:", TARGET_PATH)
print("Salida:", OUT)


## 1. Recuperación y variables de grupo

Se reutiliza el ZIP oficial almacenado en Drive. El identificador de hogar se conserva para remuestrear conjuntamente todos sus viajes. Se analizan `Mujer`, `Hombre` y los estratos 1 a 6; otras etiquetas o valores faltantes se auditan, pero no se reasignan.


In [ ]:
def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def valid_zip(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0 and zipfile.is_zipfile(path)


def upload_archive(destination):
    from google.colab import files

    print("Descargue el ZIP en el navegador si no lo conserva:")
    print(SOURCE_URL)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Suba exactamente un archivo ZIP.")
    name, content = next(iter(uploaded.items()))
    temporary = destination.with_suffix(".zip.upload")
    temporary.write_bytes(content)
    del content
    if not valid_zip(temporary):
        temporary.unlink(missing_ok=True)
        raise ValueError(f"{name} no es un ZIP válido.")
    temporary.replace(destination)
    return "manual_upload"


archive = RAW / "processed_edoh.zip"
drive_archive = DRIVE_BASE / "processed_edoh.zip"
if valid_zip(archive):
    retrieval_method = "runtime_cache"
elif valid_zip(drive_archive):
    shutil.copy2(drive_archive, archive)
    retrieval_method = "google_drive_cache"
else:
    retrieval_method = upload_archive(archive)
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, drive_archive)


def extract_recursive(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    queue = [(Path(zip_path), destination)]
    visited = set()
    while queue:
        current, target = queue.pop(0)
        signature = (str(current.resolve()), str(target.resolve()))
        if signature in visited:
            continue
        visited.add(signature)
        with zipfile.ZipFile(current) as zipped:
            zipped.extractall(target)
        for nested in target.rglob("*.zip"):
            nested_target = nested.parent / nested.stem
            nested_signature = (str(nested.resolve()), str(nested_target.resolve()))
            if nested_signature not in visited:
                queue.append((nested, nested_target))


extract_recursive(archive, INTERIM / "processed_edoh")
source_manifest = pd.DataFrame([{
    "source_id": "processed_edoh",
    "url": SOURCE_URL,
    "path": str(archive),
    "bytes": int(archive.stat().st_size),
    "sha256": sha256(archive),
    "retrieval_method": retrieval_method,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}])
source_manifest.to_csv(OUT / "source_manifest.csv", index=False)
display(source_manifest)


In [ ]:
def normalize(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(char for char in value if not unicodedata.combining(char))
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def clean_key(series):
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.str.replace(r"\.0$", "", regex=True)
    invalid = cleaned.map(normalize).isin(
        ["", "nan", "none", "na", "no_aplica", "sin_informacion"]
    )
    return cleaned.mask(invalid)


def canonical_utam(series):
    cleaned = clean_key(series)

    def canonical(value):
        if pd.isna(value):
            return pd.NA
        text = str(value).strip().upper().replace(" ", "")
        upr = re.fullmatch(r"UPR0*(\d+)", text)
        if upr:
            return f"UPR{int(upr.group(1)):04d}"
        utam = re.fullmatch(r"(?:UTAM)?0*(\d+)", text)
        if utam:
            return f"UTAM{int(utam.group(1)):03d}"
        return text

    return cleaned.map(canonical).astype("string")


def parse_number(series):
    raw = series.astype("string").str.strip()
    comma_share = float(raw.str.contains(",", regex=False, na=False).mean())
    if comma_share > 0.05:
        raw = raw.str.replace(".", "", regex=False)
        raw = raw.str.replace(",", ".", regex=False)
    return pd.to_numeric(raw, errors="coerce")


def csv_format(path):
    for encoding in ("utf-8-sig", "latin-1"):
        try:
            with path.open("r", encoding=encoding, errors="strict") as handle:
                sample = handle.read(32768)
            delimiter = csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
            return encoding, delimiter
        except (UnicodeDecodeError, csv.Error):
            continue
    raise ValueError(f"No fue posible detectar el formato: {path}")


csv_paths = list((INTERIM / "processed_edoh").rglob("*.csv"))
trip_paths = [
    path for path in csv_paths
    if "viaje" in normalize(path.stem) and "etapa" not in normalize(path.stem)
]
if not trip_paths:
    raise FileNotFoundError("No se encontró el módulo CSV de viajes.")
trip_path = sorted(trip_paths, key=lambda path: (len(path.parts), len(path.name)))[0]

encoding, delimiter = csv_format(trip_path)
header = pd.read_csv(trip_path, sep=delimiter, encoding=encoding, nrows=0)
column_map = {normalize(column): column for column in header.columns}
required = ["key_hg", "utam_ori", "utam_des", "fexp_vj", "sexo", "estra_hg"]
missing_columns = sorted(set(required) - set(column_map))
if missing_columns:
    raise KeyError(f"Faltan columnas: {missing_columns}")

trips = pd.read_csv(
    trip_path,
    sep=delimiter,
    encoding=encoding,
    usecols=[column_map[column] for column in required],
    dtype="string",
    low_memory=False,
)
trips.columns = [normalize(column) for column in trips.columns]
trips["household"] = clean_key(trips["key_hg"])
trips["utam_ori"] = canonical_utam(trips["utam_ori"])
trips["utam_des"] = canonical_utam(trips["utam_des"])
trips["weight"] = parse_number(trips["fexp_vj"])
trips["sex_group"] = trips["sexo"].map(normalize)
trips["stratum_group"] = clean_key(trips["estra_hg"])
trips = trips[
    trips[["household", "utam_ori", "utam_des"]].notna().all(axis=1)
    & trips["weight"].gt(0)
].reset_index(drop=True)

print("Filas con hogar, zonas y peso válidos:", f"{len(trips):,}")


## 2. Kernel poblacional y conjuntos objetivo

El orden de nodos y la matriz poblacional provienen de Gate 4. Los destinos provienen de Gate 6. Ningún destino se redefine usando los resultados de los grupos.


In [ ]:
edges = pd.read_parquet(EDGE_PATH).copy()
for column in ("utam_ori", "utam_des"):
    edges[column] = canonical_utam(edges[column])
edges = edges.dropna(subset=["utam_ori", "utam_des"])
nodes = sorted(set(edges["utam_ori"]) | set(edges["utam_des"]))
node_to_index = {node: index for index, node in enumerate(nodes)}
n_nodes = len(nodes)

P0 = np.zeros((n_nodes, n_nodes), dtype=float)
W0 = np.zeros((n_nodes, n_nodes), dtype=float)
for row in edges.itertuples(index=False):
    i = node_to_index[row.utam_ori]
    j = node_to_index[row.utam_des]
    P0[i, j] = float(row.transition_probability)
    W0[i, j] = float(row.weighted_trips)
population_outflow = W0.sum(axis=1)

valid_node = trips["utam_ori"].isin(node_to_index) & trips["utam_des"].isin(node_to_index)
trips = trips.loc[valid_node].reset_index(drop=True)
origin_index = trips["utam_ori"].map(node_to_index).to_numpy(dtype=int)
destination_index = trips["utam_des"].map(node_to_index).to_numpy(dtype=int)
base_weight = trips["weight"].to_numpy(dtype=float)

expected_groups = {
    "sex:mujer": ("sex", "mujer", trips["sex_group"].eq("mujer").to_numpy()),
    "sex:hombre": ("sex", "hombre", trips["sex_group"].eq("hombre").to_numpy()),
}
for level in map(str, range(1, 7)):
    expected_groups[f"stratum:{level}"] = (
        "stratum", level, trips["stratum_group"].eq(level).to_numpy()
    )

group_inventory = pd.DataFrame([
    {
        "group_id": group_id,
        "dimension": dimension,
        "level": level,
        "sampled_trips": int(mask.sum()),
        "weighted_trips": float(trips.loc[mask, "weight"].sum()),
        "households": int(trips.loc[mask, "household"].nunique()),
    }
    for group_id, (dimension, level, mask) in expected_groups.items()
])
group_inventory.to_csv(OUT / "group_inventory.csv", index=False)
display(group_inventory)

target_table = pd.read_csv(TARGET_PATH, dtype={"utam": "string"})
target_table["utam"] = target_table["utam"].astype("string").str.strip().str.upper()
target_sets = {}
for purpose in PURPOSES:
    for definition in TARGET_DEFINITIONS:
        codes = set(target_table.loc[
            target_table["purpose_group"].eq(purpose)
            & target_table["target_definition"].eq(definition), "utam"
        ])
        missing_codes = sorted(codes - set(nodes))
        if missing_codes:
            raise RuntimeError(f"Destinos fuera del kernel para {purpose}/{definition}: {missing_codes}")
        target_sets[(purpose, definition)] = np.array(
            sorted(node_to_index[code] for code in codes), dtype=int
        )

print("Viajes analíticos:", f"{len(trips):,}")
print("Nodos:", n_nodes)


## 3. Regularización fila a fila

El tamaño efectivo de Kish determina cuánto se conserva del kernel empírico de cada grupo. Una fila sin observaciones usa completamente el kernel poblacional. Como (\\tau>0), toda fila conserva soporte poblacional y la cadena regularizada permanece comunicante.


In [ ]:
def regularized_kernel(mask, tau, row_multiplier=None):
    selected = np.flatnonzero(mask)
    original_weights = base_weight[selected].copy()
    if row_multiplier is not None:
        multiplicity = row_multiplier[selected].astype(float)
    else:
        multiplicity = np.ones(len(selected), dtype=float)
    weights = original_weights * multiplicity
    squared_contribution = original_weights**2 * multiplicity
    origins = origin_index[selected]
    destinations = destination_index[selected]

    positive = weights > 0
    weights = weights[positive]
    squared_contribution = squared_contribution[positive]
    origins = origins[positive]
    destinations = destinations[positive]

    W = np.zeros((n_nodes, n_nodes), dtype=float)
    np.add.at(W, (origins, destinations), weights)
    out = W.sum(axis=1)
    sum_sq = np.bincount(origins, weights=squared_contribution, minlength=n_nodes)
    ess = np.divide(out**2, sum_sq, out=np.zeros(n_nodes), where=sum_sq > 0)

    empirical = np.zeros_like(W)
    observed_rows = out > 0
    empirical[observed_rows] = W[observed_rows] / out[observed_rows, None]
    omega = ess / (ess + float(tau))
    regularized = omega[:, None] * empirical + (1.0 - omega[:, None]) * P0
    return regularized, ess, omega, out


def first_passage(P, targets):
    target_mask = np.zeros(n_nodes, dtype=bool)
    target_mask[targets] = True
    transient = np.flatnonzero(~target_mask)
    Q = P[np.ix_(transient, transient)]
    mean = np.linalg.solve(np.eye(len(transient)) - Q, np.ones(len(transient)))
    full = np.zeros(n_nodes)
    full[transient] = mean
    initial = population_outflow[transient] / population_outflow[transient].sum()
    aggregate = float(initial @ mean)
    return full, aggregate


kernel_diagnostics = []
group_kernels = {}
point_rows = []
origin_rows = []
for group_id, (dimension, level, mask) in expected_groups.items():
    P_group, ess, omega, group_outflow = regularized_kernel(mask, PRIMARY_TAU)
    group_kernels[group_id] = P_group
    kernel_diagnostics.append({
        "group_id": group_id,
        "dimension": dimension,
        "level": level,
        "rows_without_group_transitions": int((group_outflow == 0).sum()),
        "median_row_ess_observed": float(np.median(ess[ess > 0])),
        "median_omega_observed": float(np.median(omega[ess > 0])),
        "minimum_population_pooling_share": float((1.0 - omega).min()),
        "max_row_sum_error": float(np.max(np.abs(P_group.sum(axis=1) - 1))),
    })
    for (purpose, definition), targets in target_sets.items():
        by_origin, aggregate = first_passage(P_group, targets)
        point_rows.append({
            "group_id": group_id,
            "dimension": dimension,
            "level": level,
            "purpose_group": purpose,
            "target_definition": definition,
            "tau": float(PRIMARY_TAU),
            "common_origin_mean_fpt_steps": aggregate,
        })
        for index, node in enumerate(nodes):
            origin_rows.append({
                "group_id": group_id,
                "dimension": dimension,
                "level": level,
                "purpose_group": purpose,
                "target_definition": definition,
                "utam": node,
                "mean_fpt_steps": float(by_origin[index]),
            })

kernel_diagnostics = pd.DataFrame(kernel_diagnostics)
point_estimates = pd.DataFrame(point_rows)
group_fpt_by_origin = pd.DataFrame(origin_rows)
kernel_diagnostics.to_csv(OUT / "regularization_diagnostics.csv", index=False)
point_estimates.to_csv(OUT / "group_fpt_point_estimates.csv", index=False)
group_fpt_by_origin.to_parquet(OUT / "group_fpt_by_origin.parquet", index=False)
display(kernel_diagnostics)
display(point_estimates)


## 4. Sensibilidad al parámetro de contracción

Se comparan (\tau=10,20,50). La sensibilidad se mide tanto en la media común de FPT como en el ordenamiento de los orígenes frente a la especificación principal.


In [ ]:
tau_rows = []
for group_id, (dimension, level, mask) in expected_groups.items():
    primary = group_kernels[group_id]
    for tau in TAU_GRID:
        P_tau, _, _, _ = regularized_kernel(mask, tau)
        for (purpose, definition), targets in target_sets.items():
            primary_origin, primary_aggregate = first_passage(primary, targets)
            tau_origin, tau_aggregate = first_passage(P_tau, targets)
            target_mask = np.zeros(n_nodes, dtype=bool)
            target_mask[targets] = True
            eligible = ~target_mask
            rho = float(spearmanr(
                primary_origin[eligible], tau_origin[eligible]
            ).statistic)
            tau_rows.append({
                "group_id": group_id,
                "dimension": dimension,
                "level": level,
                "purpose_group": purpose,
                "target_definition": definition,
                "tau": float(tau),
                "mean_fpt_steps": tau_aggregate,
                "relative_change_from_tau20": float(
                    (tau_aggregate - primary_aggregate) / primary_aggregate
                ),
                "spearman_vs_tau20": rho,
            })

tau_sensitivity = pd.DataFrame(tau_rows)
tau_sensitivity.to_csv(OUT / "tau_sensitivity.csv", index=False)
display(
    tau_sensitivity.groupby("tau").agg(
        max_absolute_relative_change=(
            "relative_change_from_tau20", lambda x: x.abs().max()
        ),
        minimum_spearman=("spearman_vs_tau20", "min"),
    )
)


## 5. Bootstrap Poisson por hogar

Cada réplica asigna una multiplicidad Poisson(1) a cada hogar y la aplica a todos sus viajes. Se recalculan los kernels regulares y el FPT hacia los objetivos `core_50`. Cincuenta réplicas sirven como compuerta computacional; el manuscrito final deberá aumentar este número.


In [ ]:
households = trips["household"].astype("category")
household_codes = households.cat.codes.to_numpy()
n_households = int(households.cat.categories.size)
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows = []

for replicate in range(1, BOOTSTRAP_REPLICATES + 1):
    household_multiplier = rng.poisson(1, n_households)
    row_multiplier = household_multiplier[household_codes]
    for group_id, (dimension, level, mask) in expected_groups.items():
        P_boot, _, _, _ = regularized_kernel(
            mask, PRIMARY_TAU, row_multiplier=row_multiplier
        )
        for purpose in PURPOSES:
            _, aggregate = first_passage(
                P_boot, target_sets[(purpose, "core_50")]
            )
            bootstrap_rows.append({
                "replicate": int(replicate),
                "group_id": group_id,
                "dimension": dimension,
                "level": level,
                "purpose_group": purpose,
                "mean_fpt_steps": aggregate,
            })
    if replicate % 10 == 0:
        print(f"Bootstrap {replicate}/{BOOTSTRAP_REPLICATES}")

bootstrap = pd.DataFrame(bootstrap_rows)
bootstrap.to_parquet(OUT / "group_fpt_household_bootstrap.parquet", index=False)

primary_core = point_estimates[
    point_estimates["target_definition"].eq("core_50")
].copy()
intervals = (
    bootstrap.groupby(["group_id", "dimension", "level", "purpose_group"], observed=True)
    ["mean_fpt_steps"]
    .agg(
        bootstrap_mean="mean",
        bootstrap_sd="std",
        ci_lower=lambda x: x.quantile(0.025),
        ci_upper=lambda x: x.quantile(0.975),
    )
    .reset_index()
)
intervals = primary_core.merge(
    intervals,
    on=["group_id", "dimension", "level", "purpose_group"],
    how="left",
)
intervals["relative_ci_width"] = (
    (intervals["ci_upper"] - intervals["ci_lower"])
    / intervals["common_origin_mean_fpt_steps"]
)
intervals.to_csv(OUT / "group_fpt_bootstrap_intervals.csv", index=False)
display(intervals)


## 6. Contrastes preespecificados

Para sexo se reporta `Mujer − Hombre`. Para estrato se reporta la correlación de Spearman entre el nivel 1–6 y el FPT. Los intervalos bootstrap describen incertidumbre; la compuerta no exige significancia ni una dirección determinada.


In [ ]:
contrast_rows = []
for purpose in PURPOSES:
    sex_point = primary_core[
        primary_core["dimension"].eq("sex")
        & primary_core["purpose_group"].eq(purpose)
    ].set_index("level")["common_origin_mean_fpt_steps"]
    sex_difference = float(sex_point["mujer"] - sex_point["hombre"])

    strata_point = primary_core[
        primary_core["dimension"].eq("stratum")
        & primary_core["purpose_group"].eq(purpose)
    ].copy()
    strata_point["level_numeric"] = pd.to_numeric(strata_point["level"])
    stratum_rho = float(spearmanr(
        strata_point["level_numeric"],
        strata_point["common_origin_mean_fpt_steps"],
    ).statistic)

    sex_boot = bootstrap[
        bootstrap["dimension"].eq("sex")
        & bootstrap["purpose_group"].eq(purpose)
    ].pivot(index="replicate", columns="level", values="mean_fpt_steps")
    sex_boot_difference = sex_boot["mujer"] - sex_boot["hombre"]

    stratum_boot_values = []
    for _, frame in bootstrap[
        bootstrap["dimension"].eq("stratum")
        & bootstrap["purpose_group"].eq(purpose)
    ].groupby("replicate"):
        ordered = frame.assign(
            level_numeric=pd.to_numeric(frame["level"])
        ).sort_values("level_numeric")
        stratum_boot_values.append(float(spearmanr(
            ordered["level_numeric"], ordered["mean_fpt_steps"]
        ).statistic))

    contrast_rows.extend([
        {
            "purpose_group": purpose,
            "contrast": "female_minus_male_steps",
            "point_estimate": sex_difference,
            "ci_lower": float(sex_boot_difference.quantile(0.025)),
            "ci_upper": float(sex_boot_difference.quantile(0.975)),
        },
        {
            "purpose_group": purpose,
            "contrast": "spearman_stratum_vs_fpt",
            "point_estimate": stratum_rho,
            "ci_lower": float(np.quantile(stratum_boot_values, 0.025)),
            "ci_upper": float(np.quantile(stratum_boot_values, 0.975)),
        },
    ])

contrasts = pd.DataFrame(contrast_rows)
contrasts.to_csv(OUT / "predefined_group_contrasts.csv", index=False)
display(contrasts)


## 7. Visualización

La figura muestra estimaciones `core_50` con intervalos percentiles del bootstrap. Los paneles de sexo y estrato se separan para evitar interpretar sus escalas categóricas como equivalentes.


In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

sex_plot = intervals[intervals["dimension"].eq("sex")].copy()
sex_order = ["hombre", "mujer"]
for offset, purpose in enumerate(PURPOSES):
    subset = (
        sex_plot[sex_plot["purpose_group"].eq(purpose)]
        .set_index("level").reindex(sex_order).reset_index()
    )
    x = np.arange(len(subset)) + (offset - 1) * 0.18
    y = subset["common_origin_mean_fpt_steps"].to_numpy()
    lower = y - subset["ci_lower"].to_numpy()
    upper = subset["ci_upper"].to_numpy() - y
    axes[0].errorbar(x, y, yerr=[lower, upper], fmt="o", capsize=3, label=purpose)
axes[0].set_xticks(np.arange(len(sex_order)))
axes[0].set_xticklabels(sex_order)
axes[0].set_xlabel("Sexo")
axes[0].set_ylabel("FPT medio regularizado (transiciones)")
axes[0].legend(title="Propósito")

stratum_plot = intervals[intervals["dimension"].eq("stratum")].copy()
for purpose in PURPOSES:
    subset = stratum_plot[stratum_plot["purpose_group"].eq(purpose)].copy()
    subset["level_numeric"] = pd.to_numeric(subset["level"])
    subset = subset.sort_values("level_numeric")
    axes[1].plot(
        subset["level_numeric"], subset["common_origin_mean_fpt_steps"],
        marker="o", label=purpose,
    )
    axes[1].fill_between(
        subset["level_numeric"].to_numpy(dtype=float),
        subset["ci_lower"].to_numpy(dtype=float),
        subset["ci_upper"].to_numpy(dtype=float), alpha=0.15,
    )
axes[1].set_xticks(range(1, 7))
axes[1].set_xlabel("Estrato")
axes[1].set_ylabel("FPT medio regularizado (transiciones)")
axes[1].legend(title="Propósito")

fig.tight_layout()
fig.savefig(OUT / "regularized_group_fpt.png", dpi=180, bbox_inches="tight")
plt.show()


## 8. Compuerta 8

La compuerta evalúa viabilidad numérica y reproducibilidad, no si las diferencias son estadísticamente significativas. Una ausencia de diferencia también es un resultado válido.


In [ ]:
tau_nonprimary = tau_sensitivity[~tau_sensitivity["tau"].eq(PRIMARY_TAU)]
thresholds = {
    "minimum_sampled_trips_per_group": 500,
    "maximum_kernel_row_sum_error": 1e-10,
    "minimum_tau_sensitivity_spearman": 0.90,
    "maximum_tau_relative_change": 0.20,
    "maximum_relative_ci_width": 0.60,
}

checks = {
    "gate7_dependency_valid": bool(gate7_status.get("gate_7_passed", False)),
    "expected_groups_present": bool(len(group_inventory) == 8),
    "group_sample_size_valid": bool(
        group_inventory["sampled_trips"].ge(
            thresholds["minimum_sampled_trips_per_group"]
        ).all()
    ),
    "regularized_kernels_stochastic": bool(
        kernel_diagnostics["max_row_sum_error"].le(
            thresholds["maximum_kernel_row_sum_error"]
        ).all()
    ),
    "population_support_retained": bool(
        kernel_diagnostics["minimum_population_pooling_share"].gt(0).all()
    ),
    "point_estimates_finite": bool(
        np.isfinite(point_estimates["common_origin_mean_fpt_steps"]).all()
    ),
    "tau_rankings_stable": bool(
        tau_nonprimary["spearman_vs_tau20"].ge(
            thresholds["minimum_tau_sensitivity_spearman"]
        ).all()
    ),
    "tau_magnitude_stable": bool(
        tau_nonprimary["relative_change_from_tau20"].abs().le(
            thresholds["maximum_tau_relative_change"]
        ).all()
    ),
    "bootstrap_complete": bool(
        bootstrap["replicate"].nunique() == BOOTSTRAP_REPLICATES
        and len(bootstrap) == BOOTSTRAP_REPLICATES * len(expected_groups) * len(PURPOSES)
    ),
    "bootstrap_intervals_finite": bool(
        np.isfinite(intervals[["ci_lower", "ci_upper"]].to_numpy()).all()
    ),
    "bootstrap_precision_valid": bool(
        intervals["relative_ci_width"].le(
            thresholds["maximum_relative_ci_width"]
        ).all()
    ),
}
checks["gate_8_passed"] = bool(all(checks.values()))

report = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "source_sha256": str(source_manifest.loc[0, "sha256"]),
    "estimand": "group kernel contrast under common population origin distribution",
    "regularization": {
        "primary_tau": float(PRIMARY_TAU),
        "tau_grid": [float(value) for value in TAU_GRID],
        "weight": "row Kish effective sample size",
    },
    "bootstrap": {
        "method": "Poisson(1) household cluster bootstrap",
        "replicates": int(BOOTSTRAP_REPLICATES),
        "random_seed": int(RANDOM_SEED),
        "targets": "core_50",
    },
    "thresholds": {key: float(value) for key, value in thresholds.items()},
    "group_inventory": json.loads(group_inventory.to_json(orient="records")),
    "contrasts": json.loads(contrasts.to_json(orient="records")),
    "gate": checks,
}
(OUT / "gate_8.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)
pd.Series(checks, name="result").to_csv(OUT / "gate_8_checks.csv")
display(pd.Series(checks, name="result"))

persistent_out = DRIVE_BASE / "results/gate8"
persistent_out.mkdir(parents=True, exist_ok=True)
for artifact in OUT.iterdir():
    if artifact.is_file():
        shutil.copy2(artifact, persistent_out / artifact.name)

print("Compuerta 8:", "APROBADA" if checks["gate_8_passed"] else "NO APROBADA")
print("Copia persistente:", persistent_out)


## Interpretación y siguiente paso

- Los contrastes usan un origen poblacional común: describen diferencias en patrones de movilidad, no desigualdad residencial total.
- Los intervalos con 50 réplicas constituyen una compuerta; el análisis final deberá usar al menos 500 réplicas y conservar el remuestreo por hogar.
- No debe interpretarse una asociación por estrato como causal.
- Si la Compuerta 8 pasa, el siguiente análisis incorporará la distribución residencial de cada grupo y descompondrá la brecha total en composición espacial y diferencias del kernel.
